# 01 · BMV data pipeline: from asset list to PostgreSQL

**Setup:** follow `README.md`: install dependencies, create an empty PostgreSQL database
named `bmv_quant`, and configure `.env` in the project root. Select that environment's
kernel and choose **Run All**. The notebook creates tables automatically.
Internet access is required for Yahoo Finance.

Flow: Actinver files → asset catalog → download → PostgreSQL → validation.
The first run loads from `HISTORY_START`; subsequent runs download from the latest
stored date **for each asset**. Tables are never deleted and IDs are never reset.
Yahoo may not provide data for every candidate `.MX` symbol.

In [2]:
from pathlib import Path
from datetime import date, timedelta
import os
import logging
import time

import numpy as np
import pandas as pd
import yfinance as yf
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, MetaData, Table, inspect, or_
from sqlalchemy.engine import URL
from sqlalchemy.dialects.postgresql import insert

logging.getLogger('yfinance').setLevel(logging.CRITICAL)
# Works with the kernel started from the root, notebooks/, or a subdirectory.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'sql/schema.sql').is_file()
                     and (p / 'data//raw/datosReto.txt').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the notebook from the project directory.')
DATA_DIR = PROJECT_ROOT / 'data'
if not (PROJECT_ROOT / '.env').is_file():
    raise RuntimeError('Copy .env.example to .env and configure PostgreSQL before Run All.')
load_dotenv(PROJECT_ROOT / '.env', override=False)
required = ['POSTGRES_USER', 'POSTGRES_PASSWORD', 'POSTGRES_HOST', 'POSTGRES_PORT', 'POSTGRES_DB']
missing = [key for key in required if not os.getenv(key)]
if missing or os.getenv('POSTGRES_PASSWORD') == 'your_password_here':
    raise RuntimeError('Configure the PostgreSQL variables in .env; do not print their values.')
HISTORY_START = date.fromisoformat(os.getenv('HISTORY_START', '2020-01-01'))
if HISTORY_START >= date.today():
    raise ValueError('HISTORY_START must be earlier than today.')
RETRY_UNAVAILABLE = os.getenv('RETRY_UNAVAILABLE', 'true').lower() == 'true'

# Simple maintenance policy, in calendar days.
OVERLAP_DAYS = 7
HISTORY_REFRESH_DAYS = 30

## 1. Secure connection and schema
`URL.create` handles passwords containing special characters without concatenating them
into code. We do not display the connection URL or driver errors that might reveal
connection details. The schema protects asset/price relationships and `(asset_id, date)`
uniqueness. If you already have experimental tables with different columns, use an empty
database: this notebook does not migrate or delete them.

For databases created with this project's previous schema, an additive
`ALTER TABLE ... ADD COLUMN IF NOT EXISTS` introduces `prices.quality_flags`.
Existing rows remain unchanged and receive an empty flag by default. Older experimental
schemas still require separate migration or an empty database.

The schema also adds `assets.history_refreshed_on` and a `price_rejections` table.
These track maintenance and unresolved dates without deleting existing prices.

In [2]:
try:
    engine = create_engine(URL.create(
        'postgresql+psycopg2', username=os.environ['POSTGRES_USER'],
        password=os.environ['POSTGRES_PASSWORD'], host=os.environ['POSTGRES_HOST'],
        port=int(os.environ['POSTGRES_PORT']), database=os.environ['POSTGRES_DB']),
        pool_pre_ping=True, hide_parameters=True)
    with engine.begin() as conn:
        for statement in (PROJECT_ROOT / 'sql/schema.sql').read_text(encoding='utf-8').split(';'):
            if statement.strip():
                conn.execute(text(statement))
    expected = {
        'assets': {'asset_id','code','base_symbol','series','market_code','company_name',
                   'asset_type','actinver_symbol','yahoo_symbol','data_available','history_refreshed_on'},
        'prices': {'date','asset_id','open','high','low','close','adjusted_close','volume','quality_flags'},
    }
    inspector = inspect(engine)
    for name, columns in expected.items():
        actual = {c['name'] for c in inspector.get_columns(name, schema='public')}
        if not columns <= actual:
            raise ValueError('Incompatible schema')
except Exception:
    raise RuntimeError('Could not initialize PostgreSQL. Check the service, .env, permissions, and schema; '
                       'if you have experimental tables, use an empty database.') from None
metadata = MetaData()
assets_table = Table('assets', metadata, schema='public', autoload_with=engine)
prices_table = Table('prices', metadata, schema='public', autoload_with=engine)
print('Connection and tables are ready.')
rejections_table = Table('price_rejections', metadata, schema='public', autoload_with=engine)

Connection and tables are ready.


## 2. Reading and building the asset universe
We validate the format before writing to SQL. A one-to-one join on symbol and series
prevents row multiplication. Missing descriptions are allowed.
Yahoo symbols are candidates: we do not assume that every instrument is available.

In [3]:
def read_pipe_file(path, width):
    rows = [line.strip().split('|') for line in path.read_text(encoding='utf-8-sig').splitlines()
            if line.strip()]
    if not rows or any(len(row) != width for row in rows):
        raise ValueError(f'Unexpected format in {path.name}: expected {width} columns.')
    return pd.DataFrame(rows).map(lambda value: value.strip())


def build_asset_master(data_dir):
    universe = read_pipe_file(data_dir / 'datosReto.txt', 22).rename(
        columns={0: 'code', 1: 'base_symbol', 2: 'series'})
    desc = read_pipe_file(data_dir / 'datosRetoDescripcion.txt', 4)
    desc.columns = ['market_code', 'base_symbol', 'series', 'company_name']
    keys = ['base_symbol', 'series']
    for frame in [universe, desc]:
        if frame.duplicated(keys).any() or frame[keys].eq('').any().any():
            raise ValueError('The universe contains empty or duplicate symbol/series keys.')
    master = universe[['code', *keys]].merge(desc, on=keys, how='left', validate='one_to_one')
    types = {'1':'STOCK', '1A':'STOCK', '41':'STOCK', 'CF':'STOCK',
             '1B':'ETF', '1I':'ETF', '52':'FUND'}
    master['asset_type'] = master['code'].map(types).fillna('UNKNOWN')
    master['actinver_symbol'] = master['base_symbol'] + master['series'].replace('*', '')
    master['yahoo_symbol'] = master['actinver_symbol'] + '.MX'
    return master

asset_master = build_asset_master(DATA_DIR)
print(f'Universe: {len(asset_master)} assets.')
display(asset_master.head())

Universe: 226 assets.


,code,base_symbol,series,market_code,company_name,asset_type,actinver_symbol,yahoo_symbol
0,1A,AA1,*,NaN,NaN,STOCK,AA1,AA1.MX
1,1A,AAL,*,1A,AMERICAN AIRLINES GROUP INC.,STOCK,AAL,AAL.MX
2,1A,AAPL,*,1A,APPLE COMPUTER INC,STOCK,AAPL,AAPL.MX
3,1I,AAXJ,*,1I,ISHARES MSCI ALL COUNTRY ASIA EX JAPAN ETF,ETF,AAXJ,AAXJ.MX
4,1A,ABBV,*,NaN,NaN,STOCK,ABBV,ABBV.MX


## 3. Persistent asset catalog
We insert new assets and update descriptions while preserving `asset_id` and
`data_available`. Assets with existing history do not undergo a separate bulk availability
check. Assets removed from the source file retain their prices, but this run downloads
only assets included in the current universe.

In [4]:
def sync_assets(engine, master):
    records = master.astype(object).where(pd.notna(master), None).to_dict('records')
    stmt = insert(assets_table).values(records)
    fields = ['code','market_code','company_name','asset_type','actinver_symbol']
    with engine.begin() as conn:
        conn.execute(stmt.on_conflict_do_update(
            index_elements=['base_symbol','series'],
            set_={key: stmt.excluded[key] for key in fields}))
    # Also preserve Yahoo symbols corrected manually in SQL.
    with engine.connect() as conn:
        stored = pd.read_sql(text('SELECT * FROM public.assets'), conn)
    return master[['base_symbol','series']].merge(
        stored, on=['base_symbol','series'], validate='one_to_one')

current_assets = sync_assets(engine, asset_master)

## 4. Downloading, validating, and saving prices

We request OHLC and `Adj Close` with `auto_adjust=False`. Return calculations use
`adjusted_close`; `close` is retained for price and turnover information.

Missing/non-finite values, nonpositive closes, and invalid volumes are recorded as rejected
dates. Usable observations are saved. Nonpositive Open/High/Low or inconsistent OHLC
relationships are preserved with explicit warnings. We never invent replacement prices.
Malformed dates or conflicting observations fail the response because they cannot be
safely associated with a single date. Responses outside `[start, end)` are filtered.

The helpers below have separate jobs: normalize data, download it, compare adjustment
bases, and save prices together with their unresolved dates. Saving is transactional.

In [5]:
PRICE_COLUMNS = [
    'date', 'asset_id', 'open', 'high', 'low', 'close',
    'adjusted_close', 'volume', 'quality_flags'
]


def normalize_prices(raw, asset_id, start, end):
    """Keep usable observations and attach rejected dates for later recovery."""
    if raw is None or raw.empty:
        return pd.DataFrame(columns=PRICE_COLUMNS)
    data = raw.copy()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    data.columns = data.columns.str.lower()
    data = data.rename(columns={'adj close': 'adjusted_close'})
    needed = ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']
    if not set(needed) <= set(data.columns):
        raise ValueError('Missing OHLCV or adjusted-close columns')
    data = data[needed].copy()
    dates = pd.to_datetime(data.index, errors='coerce').tz_localize(None)
    if dates.isna().any():
        raise ValueError('Malformed dates: the download cannot be safely persisted')
    data['date'] = dates.date
    data['asset_id'] = int(asset_id)
    data = data.loc[(data['date'] >= start) & (data['date'] < end)].copy()
    if data.empty:
        return pd.DataFrame(columns=PRICE_COLUMNS)
    for column in needed:
        data[column] = pd.to_numeric(data[column], errors='coerce')

    # Conflicting duplicates are ambiguous, so do not pick one silently.
    data = data.drop_duplicates()
    if data.duplicated(['asset_id', 'date']).any():
        raise ValueError('Conflicting observations for the same date')
    checks = pd.DataFrame({
        'missing_or_nonfinite_value': np.isfinite(data[needed]).all(axis=1),
        'nonpositive_close': (data[['close', 'adjusted_close']] > 0).all(axis=1),
        'invalid_volume': (data['volume'] >= 0) & (data['volume'] % 1 == 0)
                          & (data['volume'] < 2**63),
    }, index=data.index)
    valid = checks.all(axis=1)
    rejected = []
    for position in np.flatnonzero(~valid.to_numpy()):
        rejected.append({
            'asset_id': int(asset_id),
            'date': data.iloc[position]['date'],
            'reason': '|'.join(checks.columns[~checks.iloc[position]]),
        })
    data = data.loc[valid].copy()

    # Nonpositive OHL and inconsistent candles remain visible warnings.
    data['quality_flags'] = ''
    data.loc[(data[['open', 'high', 'low']] <= 0).any(axis=1), 'quality_flags'] += '|zero_or_negative_ohl'
    data.loc[data['high'] < data[['open', 'close', 'low']].max(axis=1), 'quality_flags'] += '|high_below_other_ohlc'
    data.loc[data['low'] > data[['open', 'close', 'high']].min(axis=1), 'quality_flags'] += '|low_above_other_ohlc'
    data['quality_flags'] = data['quality_flags'].str.strip('|')
    data['volume'] = data['volume'].astype('int64')
    result = data[PRICE_COLUMNS].reset_index(drop=True)
    # attrs travels with this frame into insert_prices, including when all rows failed.
    result.attrs['rejected_rows'] = rejected
    return result


def download_prices(symbol, asset_id, start, end):
    """Retry provider failures, but report deterministic validation failures once."""
    for attempt in range(3):
        try:
            raw = yf.download(
                symbol, start=start.isoformat(), end=end.isoformat(), interval='1d',
                auto_adjust=False, progress=False, threads=False,
                timeout=20, multi_level_index=False)
        except Exception as exc:
            status = f'download failure ({type(exc).__name__})'
        else:
            try:
                frame = normalize_prices(raw, asset_id, start, end)
            except ValueError as exc:
                return pd.DataFrame(columns=PRICE_COLUMNS), f'validation failure: {exc}'
            except Exception as exc:
                return pd.DataFrame(columns=PRICE_COLUMNS), f'normalization failure ({type(exc).__name__})'
            rejected_count = len(frame.attrs.get('rejected_rows', []))
            if not frame.empty or rejected_count:
                warning_count = int(frame['quality_flags'].ne('').sum())
                return frame, (f'downloaded | {rejected_count} rejected row(s) | '
                               f'{warning_count} OHLC warning row(s)')
            if raw is not None and not raw.empty:
                return frame, 'no new rows in requested date window'
            status = 'no data (symbol, calendar, or provider)'
        if attempt < 2:
            time.sleep(2 ** attempt)
    return pd.DataFrame(columns=PRICE_COLUMNS), status


def adjustments_changed(engine, frame, asset_id):
    """Compare overlapping dates before combining old and new adjusted prices."""
    if frame.empty:
        return False
    with engine.connect() as conn:
        stored = pd.read_sql(text("""
            SELECT date, adjusted_close FROM public.prices
            WHERE asset_id = :asset_id AND date BETWEEN :start AND :end
        """), conn, params={
            'asset_id': int(asset_id), 'start': min(frame['date']), 'end': max(frame['date'])})
    stored['date'] = pd.to_datetime(stored['date']).dt.date
    overlap = frame[['date', 'adjusted_close']].merge(
        stored, on='date', suffixes=('_new', '_old'), validate='one_to_one')
    if overlap.empty:
        # Without an anchor, a full refresh is safer than joining incompatible histories.
        return True
    old = pd.to_numeric(overlap['adjusted_close_old'], errors='coerce').to_numpy(dtype=float)
    new = overlap['adjusted_close_new'].to_numpy(dtype=float)
    return not np.isclose(old, new, rtol=1e-6, atol=1e-8).all()


def insert_prices(engine, frame, asset_id, full_refresh=False):
    """Persist prices and unresolved dates together in one transaction."""
    rejected = list(frame.attrs.get('rejected_rows', []))
    if frame.empty and not rejected:
        return 0
    records = frame.to_dict('records')
    affected = 0
    with engine.begin() as conn:
        if full_refresh and records:
            # A partial provider response must not leave old adjustment values unnoticed.
            stored_dates = set(conn.execute(text(
                'SELECT date FROM public.prices WHERE asset_id = :asset_id'
            ), {'asset_id': int(asset_id)}).scalars())
            returned_dates = {record['date'] for record in records + rejected}
            rejected.extend({'asset_id': int(asset_id), 'date': day,
                             'reason': 'missing_from_history_response'}
                            for day in sorted(stored_dates - returned_dates))
        for offset in range(0, len(records), 1000):
            batch = records[offset:offset + 1000]
            stmt = insert(prices_table).values(batch)
            fields = ['open', 'high', 'low', 'close', 'adjusted_close', 'volume', 'quality_flags']
            stmt = stmt.on_conflict_do_update(
                index_elements=['asset_id', 'date'],
                set_={key: stmt.excluded[key] for key in fields},
                where=or_(*[prices_table.c[key].is_distinct_from(stmt.excluded[key]) for key in fields]))
            affected += conn.execute(stmt).rowcount
            # Only a valid observation for this exact date resolves a prior rejection.
            conn.execute(rejections_table.delete().where(
                rejections_table.c.asset_id == int(asset_id),
                rejections_table.c.date.in_([record['date'] for record in batch])))
        for offset in range(0, len(rejected), 1000):
            stmt = insert(rejections_table).values(rejected[offset:offset + 1000])
            conn.execute(stmt.on_conflict_do_update(
                index_elements=['asset_id', 'date'], set_={'reason': stmt.excluded.reason}))
        if records:
            values = {'data_available': True}
            if full_refresh:
                values['history_refreshed_on'] = date.today()
            conn.execute(assets_table.update().where(
                assets_table.c.asset_id == int(asset_id)).values(**values))
    frame.attrs['persisted_rejections'] = len(rejected)
    return affected

## 5. Initial load, incremental update, and history recovery

Read `plan_download` as a short decision list:

1. No prices: download from `HISTORY_START`.
2. Missing adjusted prices: download and update the complete stored history.
3. Rejected dates: retry the history and remove a rejection only when that date is valid.
4. No recorded refresh, or 30 days since the last one: refresh the complete history.
5. Otherwise: download from seven calendar days before the last stored observation.

The overlap checks whether adjusted prices changed. If they did, we replace that response
with a full-history download before saving. This prevents joining new adjustments to an
old adjustment basis. Arbitrary older provider corrections that do not affect the overlap
are picked up by the scheduled full refresh, not necessarily immediately.

`HISTORY_REFRESH_DAYS` and `OVERLAP_DAYS` are the two maintenance settings in the first code
cell. A first run after upgrading may be slow because existing adjusted prices need recovery.
The upsert updates existing dates and inserts new ones without creating duplicate keys.
A separate `price_rejections` table retains invalid dates for the next run. A missing or
empty provider response never clears a pending rejection. The screener excludes assets
with unresolved rejections until their data is recovered.

`RETRY_UNAVAILABLE=false` skips previously unavailable assets without prices, but does not
skip recovery of dates already recorded as rejected.

In [6]:
def plan_download(last_date, first_date, missing_adjusted, pending_rejections,
                  history_refreshed_on, history_start, end):
    """Choose a full load when needed; otherwise inspect a short overlap."""
    history_floor = history_start
    if pd.notna(first_date):
        history_floor = min(history_start, pd.Timestamp(first_date).date())
    if pd.isna(last_date):
        return history_floor, 'initial'
    if missing_adjusted > 0:
        return history_floor, 'recover adjusted history'
    if pending_rejections > 0:
        return history_floor, 'retry rejected dates'
    if (pd.isna(history_refreshed_on)
        or (end - pd.Timestamp(history_refreshed_on).date()).days >= HISTORY_REFRESH_DAYS):
        return history_floor, 'scheduled history refresh'
    start = max(history_floor, pd.Timestamp(last_date).date() - timedelta(days=OVERLAP_DAYS))
    return start, 'incremental'

In [7]:
def update_market_data(engine, current_assets, history_start, retry_unavailable=True):
    # Read persisted maintenance state every time, even when rerunning only this cell.
    with engine.connect() as conn:
        price_status = pd.read_sql(text("""
            SELECT a.asset_id, a.history_refreshed_on,
                   p.first_date, p.last_date,
                   COALESCE(p.missing_adjusted, 0) AS missing_adjusted,
                   COALESCE(r.pending_rejections, 0) AS pending_rejections
            FROM public.assets a
            LEFT JOIN (
                SELECT asset_id, MIN(date) AS first_date, MAX(date) AS last_date,
                       COUNT(*) FILTER (WHERE adjusted_close IS NULL OR adjusted_close <= 0)
                           AS missing_adjusted
                FROM public.prices GROUP BY asset_id
            ) p ON p.asset_id = a.asset_id
            LEFT JOIN (
                SELECT asset_id, COUNT(*) AS pending_rejections
                FROM public.price_rejections GROUP BY asset_id
            ) r ON r.asset_id = a.asset_id
        """), conn)
        total = conn.execute(text('SELECT COUNT(*) FROM public.prices')).scalar_one()
    info = current_assets[['asset_id', 'yahoo_symbol']].merge(
        price_status, on='asset_id', how='left', validate='one_to_one')
    report = []
    end = date.today()  # Exclusive end: do not store the unfinished current session.
    for position, row in enumerate(info.itertuples(index=False), start=1):
        start, mode = plan_download(
            row.last_date, row.first_date, row.missing_adjusted, row.pending_rejections,
            row.history_refreshed_on, history_start, end)
        print(f'[{position}/{len(info)}] {row.yahoo_symbol} | {mode} | from {start}', flush=True)
        affected = warning_rows = rejected_rows = 0
        if start >= end:
            status = 'no new date window'
        elif mode == 'initial' and not retry_unavailable and total > 0 and row.pending_rejections == 0:
            status = 'no history; retry disabled'
        else:
            frame, status = download_prices(row.yahoo_symbol, row.asset_id, start, end)
            if mode == 'incremental' and adjustments_changed(engine, frame, row.asset_id):
                print('  Adjustment change detected: refreshing the complete stored history.', flush=True)
                mode = 'adjustment change refresh'
                start = min(history_start, pd.Timestamp(row.first_date).date())
                # Discard the incremental response: never join it to an incompatible history.
                frame, status = download_prices(row.yahoo_symbol, row.asset_id, start, end)
            warning_rows = int(frame['quality_flags'].ne('').sum())
            rejected_rows = len(frame.attrs.get('rejected_rows', []))
            affected = insert_prices(engine, frame, row.asset_id, full_refresh=(mode != 'incremental'))
            rejected_rows = frame.attrs.get('persisted_rejections', rejected_rows)
        print(f'  {status} | Rows affected: {affected}', flush=True)
        report.append({
            'symbol': row.yahoo_symbol, 'mode': mode, 'start': start, 'status': status,
            'rows_affected': affected, 'quality_warning_rows': warning_rows,
            'rejected_rows': rejected_rows})
    return pd.DataFrame(report)

In [8]:
update_report = update_market_data(
    engine,
    current_assets,
    HISTORY_START,
    RETRY_UNAVAILABLE
)

[1/226] AA1.MX | incremental | from 2026-09-25
  downloaded | 0 rejected row(s) | 0 OHLC warning row(s) | Rows affected: 0
[2/226] AAL.MX | incremental | from 2026-09-25
  downloaded | 0 rejected row(s) | 0 OHLC warning row(s) | Rows affected: 0
[3/226] AAPL.MX | incremental | from 2026-09-25
  downloaded | 0 rejected row(s) | 0 OHLC warning row(s) | Rows affected: 0
[4/226] AAXJ.MX | incremental | from 2026-09-25
  downloaded | 0 rejected row(s) | 0 OHLC warning row(s) | Rows affected: 0
[5/226] ABBV.MX | incremental | from 2026-09-25
  downloaded | 0 rejected row(s) | 0 OHLC warning row(s) | Rows affected: 0
[6/226] ABNB.MX | incremental | from 2026-09-25
  downloaded | 0 rejected row(s) | 0 OHLC warning row(s) | Rows affected: 0
[7/226] ACTDUALB.MX | incremental | from 2026-09-24
  downloaded | 0 rejected row(s) | 0 OHLC warning row(s) | Rows affected: 0
[8/226] ACTI500B.MX | incremental | from 2026-09-24
  downloaded | 0 rejected row(s) | 0 OHLC warning row(s) | Rows affected: 0
[9

In [9]:
print(
    'Rows affected: '
    f"{update_report['rows_affected'].sum()}"
)

Rows affected: 3223


In [10]:
display(
    update_report
    .groupby(
        [
            'mode',
            'status'
        ],
        dropna=False
    )
    .agg(
        assets=(
            'symbol',
            'count'
        ),

        rows=(
            'rows_affected',
            'sum'
        ),

        rejected_rows=('rejected_rows', 'sum'),

        quality_warning_rows=(
            'quality_warning_rows',
            'sum'
        )
    )
    .reset_index()
)

,mode,status,assets,rows,rejected_rows,quality_warning_rows
0,incremental,downloaded | 0 rejected row(s) | 0 OHLC warnin...,202,0,0,0
1,incremental,downloaded | 0 rejected row(s) | 1 OHLC warnin...,1,0,0,1
2,incremental,downloaded | 0 rejected row(s) | 2 OHLC warnin...,1,0,0,2
3,initial,"no data (symbol, calendar, or provider)",9,0,0,0
4,recover adjusted history,downloaded | 177 rejected row(s) | 0 OHLC warn...,1,77,177,0
5,retry rejected dates,downloaded | 0 rejected row(s) | 0 OHLC warnin...,2,0,2,0
6,retry rejected dates,downloaded | 0 rejected row(s) | 1 OHLC warnin...,6,0,2,6
7,retry rejected dates,downloaded | 0 rejected row(s) | 106 OHLC warn...,1,0,1,106
8,retry rejected dates,downloaded | 0 rejected row(s) | 3 OHLC warnin...,3,3146,3,9


In [11]:
# Show failures and quality issues, including singular "warning" and rejected rows.
attention = (
    update_report['status'].str.contains('no data|failure|disabled', case=False, na=False)
    | (update_report['quality_warning_rows'] > 0)
    | (update_report['rejected_rows'] > 0)
)
display(update_report.loc[attention])

,symbol,mode,start,status,rows_affected,quality_warning_rows,rejected_rows
8,ACTIAIB.MX,initial,2020-01-01,"no data (symbol, calendar, or provider)",0,0,0
10,ACTICREB.MX,initial,2020-01-01,"no data (symbol, calendar, or provider)",0,0,0
19,ACTIVARB.MX,initial,2020-01-01,"no data (symbol, calendar, or provider)",0,0,0
53,CRM.MX,retry rejected dates,2020-01-01,downloaded | 0 rejected row(s) | 3 OHLC warnin...,970,3,1
62,DINAMOB.MX,initial,2020-01-01,"no data (symbol, calendar, or provider)",0,0,0
66,ESCALAB.MX,retry rejected dates,2020-01-01,downloaded | 0 rejected row(s) | 1 OHLC warnin...,0,1,1
67,ESFERAB.MX,retry rejected dates,2020-01-01,downloaded | 0 rejected row(s) | 1 OHLC warnin...,0,1,1
92,GME.MX,incremental,2026-09-25,downloaded | 0 rejected row(s) | 1 OHLC warnin...,0,1,0
103,JPMRVUSB-1.MX,initial,2020-01-01,"no data (symbol, calendar, or provider)",0,0,0
110,LASITEB-1.MX,initial,2020-01-01,"no data (symbol, calendar, or provider)",0,0,0


## 6. Final validation
The summary shows coverage and date ranges by asset type. `data_available` means that
prices have been stored at least once; the preceding report describes this run.
Before sharing, clear outputs and save. Keep `.env` local.

In [12]:
with engine.connect() as conn:

    summary = pd.read_sql(
        text("""
            SELECT
                a.asset_type,

                COUNT(
                    DISTINCT a.asset_id
                ) AS assets,

                COUNT(
                    DISTINCT p.asset_id
                ) AS assets_with_prices,

                COUNT(p.date)
                    AS price_rows,

                COUNT(p.adjusted_close)
                    AS adjusted_price_rows,

                COUNT(p.date)
                - COUNT(p.adjusted_close)
                    AS missing_adjusted_rows,

                MIN(p.date)
                    AS first_date,

                MAX(p.date)
                    AS last_date

            FROM public.assets a

            LEFT JOIN public.prices p
                ON p.asset_id = a.asset_id

            GROUP BY
                a.asset_type

            ORDER BY
                a.asset_type
        """),
        conn
    )


display(summary)

,asset_type,assets,assets_with_prices,price_rows,adjusted_price_rows,missing_adjusted_rows,first_date,last_date
0,ETF,56,55,75911,75734,177,2020-01-02,2026-10-02
1,FUND,25,18,28233,28233,0,2020-01-02,2026-10-01
2,STOCK,145,144,238216,238216,0,2020-01-02,2026-10-02


In [13]:
with engine.connect() as conn:

    missing_adjusted = pd.read_sql(
        text("""
            SELECT
                a.actinver_symbol,
                a.asset_type,
                COUNT(*) AS missing_rows

            FROM public.prices p

            JOIN public.assets a
                ON p.asset_id = a.asset_id

            WHERE
                p.adjusted_close IS NULL

            GROUP BY
                a.actinver_symbol,
                a.asset_type

            ORDER BY
                missing_rows DESC
        """),
        conn
    )


display(missing_adjusted)

with engine.connect() as conn:
    pending_rejections = pd.read_sql(text("""
        SELECT a.actinver_symbol, r.date, r.reason
        FROM public.price_rejections r
        JOIN public.assets a ON a.asset_id = r.asset_id
        ORDER BY a.actinver_symbol, r.date
    """), conn)
display(pending_rejections)

,actinver_symbol,asset_type,missing_rows
0,SOXS,ETF,177


,actinver_symbol,date,reason
0,CRM,2026-08-10,missing_from_history_response
1,ESCALAB,2026-09-22,missing_from_history_response
2,ESFERAB,2026-09-22,missing_from_history_response
3,SOXS,2024-10-07,nonpositive_close
4,SOXS,2024-10-08,nonpositive_close
...,...,...,...
180,TSLA,2026-08-10,missing_from_history_response
181,UBER,2026-08-10,missing_from_history_response
182,UPST,2026-08-10,missing_from_history_response
183,V,2026-08-10,missing_from_history_response


In [14]:
engine.dispose()